In [213]:
import warnings
warnings.filterwarnings('ignore')

In [214]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast

In [215]:
data = pd.read_csv('transformed/past_house_results.csv')
data.head()

,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,...,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct
0,2018,Alabama,AL,False,1,89226.0,153228.0,242617,Robert Kennedy Jr,Bradley Byrne,...,39095.21,343761.59,242454.0,36.801208,63.198792,39095.21,343761.59,382856.80,10.211445,89.788555
1,2018,Alabama,AL,False,2,86931.0,138879.0,226230,Tabitha Isner,Martha Roby,...,501490.57,799289.56,225810.0,38.497409,61.502591,501490.57,799289.56,1300780.13,38.553062,61.446938
2,2018,Alabama,AL,False,3,83996.0,147770.0,231915,Mallory Hagan,Mike Rogers,...,409309.26,530372.0,231766.0,36.241727,63.758273,409309.26,530372.00,939681.26,43.558308,56.441692
3,2018,Alabama,AL,False,4,46492.0,184255.0,230969,Lee Auman,Robert Aderholt,...,61160.87,468336.48,230747.0,20.148474,79.851526,61160.87,468336.48,529497.35,11.550741,88.449259
4,2018,Alabama,AL,False,5,101388.0,159063.0,260673,Peter Joffrion,Mo Brooks,...,553386.53,1216500.38,260451.0,38.927860,61.072140,553386.53,1216500.38,1769886.91,31.266773,68.733227


In [216]:
data.columns

Index(['year', 'state', 'state_po', 'special', 'district', 'dem', 'rep',
       'totalvotes', 'dem_cand', 'rep_cand', 'dem_inc', 'rep_inc', 'dem_funds',
       'rep_funds', '2party_votes', 'dem_pct_2p', 'rep_pct_2p',
       'dem_tot_funds', 'rep_tot_funds', 'tot_funds', 'dem_funds_2p_pct',
       'rep_funds_2p_pct'],
      dtype='object')

In [217]:
joined_dfs = []

for yr in np.unique(data['year']):
    pvi = pd.read_csv(f'transformed/pvi/past_pres_results_by{yr % 2000}dist.csv')
    prev_cyc = (yr % 2000) - ((yr % 2000) % 4)
    prev_2cyc = prev_cyc - 4
    pvi['district'] = pvi['district'].map(lambda x: x[:2] + '-00' if x[3:] == 'AL' else x)
    df = data[data['year'] == yr]
    df['district'] = df['state_po'] + '-' + df['district'].map(lambda x: f'0{x}' if x < 10 else f'{x}')
    pvi = pvi[['district', f'lean_{prev_cyc}', f'lean_{prev_2cyc}']]
    df = pd.merge(left=df, right=pvi, on='district', how='left')
    df = df.rename({
        f'lean_{prev_cyc}': 'prev_lean',
        f'lean_{prev_2cyc}': 'prev2_lean'
    }, axis='columns')


    hist_gb = pd.read_csv(f'../snoutcounter-backend/averages/historical/historical_generic_ballot_{yr}.csv')
    if yr in [2018, 2020]:
        hist_appr = pd.read_csv('../snoutcounter-backend/averages/historical/historical_presidential_approval_trump_firstterm.csv')
    else:
        hist_appr = pd.read_csv('../snoutcounter-backend/averages/historical/historical_presidential_approval_biden.csv')
    gb = hist_gb.iloc[-1]['net']
    match yr:
        case 2018:
            appr = hist_appr[pd.to_datetime(hist_appr['end_date']) == pd.to_datetime('2018-11-06')]['net'].values[0]
            inc_pres = -1 # -1 for Rep, +1 for Dem
        case 2020:
            appr = hist_appr[pd.to_datetime(hist_appr['end_date']) == pd.to_datetime('2020-11-03')]['net'].values[0]
            inc_pres = -1
        case 2022:
            appr = hist_appr[pd.to_datetime(hist_appr['end_date']) == pd.to_datetime('2022-11-08')]['net'].values[0]
            inc_pres = 1
        case 2024:
            appr = hist_appr[pd.to_datetime(hist_appr['end_date']) == pd.to_datetime('2024-11-05')]['net'].values[0]
            inc_pres = 1
        case _:
            raise ValueError('Invalid year')
    df['generic_ballot_avg'] = np.full(shape=(df.shape[0],), fill_value=gb)
    df['pres_approval_avg'] = np.full(shape=(df.shape[0],), fill_value=appr)
    df['incumbent_pres'] = np.full(shape=(df.shape[0],), fill_value=inc_pres)

    ics = pd.read_csv('data/tbmics.csv')
    ics_curr = ics[(ics['Month'] == 'November') & (ics['YYYY'] == yr)]['ICS_ALL'].values[0]
    df['ics'] = np.full(shape=(df.shape[0],), fill_value=ics_curr)

    # Best to use interaction terms for presidential approval and ICS - otherwise we won't have consistent effect
    # For instance high ICS is good for Dems if Dem president is incumbent but bad for Dems if Republican president is incumbent
    df['pres_approval_avg_x_incpres'] = df['pres_approval_avg'] * df['incumbent_pres']
    df['ics_x_incpres'] = df['ics'] * df['incumbent_pres']
    
            
    joined_dfs.append(df)

mdata = pd.concat(joined_dfs, axis=0)
mdata.head()

,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,...,dem_funds_2p_pct,rep_funds_2p_pct,prev_lean,prev2_lean,generic_ballot_avg,pres_approval_avg,incumbent_pres,ics,pres_approval_avg_x_incpres,ics_x_incpres
0,2018,Alabama,AL,False,AL-01,89226.0,153228.0,242617,Robert Kennedy Jr,Bradley Byrne,...,10.211445,89.788555,-16.192783,-14.281877,-7.921785,-10.073704,-1,97.5,10.073704,-97.5
1,2018,Alabama,AL,False,AL-02,86931.0,138879.0,226230,Tabitha Isner,Martha Roby,...,38.553062,61.446938,-17.411483,-15.288962,-7.921785,-10.073704,-1,97.5,10.073704,-97.5
2,2018,Alabama,AL,False,AL-03,83996.0,147770.0,231915,Mallory Hagan,Mike Rogers,...,43.558308,56.441692,-18.001999,-14.821901,-7.921785,-10.073704,-1,97.5,10.073704,-97.5
3,2018,Alabama,AL,False,AL-04,46492.0,184255.0,230969,Lee Auman,Robert Aderholt,...,11.550741,88.449259,-33.277483,-27.698357,-7.921785,-10.073704,-1,97.5,10.073704,-97.5
4,2018,Alabama,AL,False,AL-05,101388.0,159063.0,260673,Peter Joffrion,Mo Brooks,...,31.266773,68.733227,-18.480031,-16.661366,-7.921785,-10.073704,-1,97.5,10.073704,-97.5


In [218]:
mdata.columns.values

array(['year', 'state', 'state_po', 'special', 'district', 'dem', 'rep',
       'totalvotes', 'dem_cand', 'rep_cand', 'dem_inc', 'rep_inc',
       'dem_funds', 'rep_funds', '2party_votes', 'dem_pct_2p',
       'rep_pct_2p', 'dem_tot_funds', 'rep_tot_funds', 'tot_funds',
       'dem_funds_2p_pct', 'rep_funds_2p_pct', 'prev_lean', 'prev2_lean',
       'generic_ballot_avg', 'pres_approval_avg', 'incumbent_pres', 'ics',
       'pres_approval_avg_x_incpres', 'ics_x_incpres'], dtype=object)

In [219]:
mdata.shape

(1728, 30)

In [220]:
isinstance(3, int)

True

In [221]:
for party in ['dem', 'rep']:
    mdata[f'{party}_funds_2p_pct'] = mdata[f'{party}_funds_2p_pct'].fillna(0)

In [222]:
# Exclude uncontested races and races w/ all candidates from one party
mdata['no_dem_cand_flag'] = mdata['dem_cand'].map(lambda x: x == '[]')
mdata['no_rep_cand_flag'] = mdata['rep_cand'].map(lambda x: x == '[]')
mdata = mdata[~mdata['no_dem_cand_flag']]
mdata = mdata[~mdata['no_rep_cand_flag']]
mdata.head()

,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,...,prev_lean,prev2_lean,generic_ballot_avg,pres_approval_avg,incumbent_pres,ics,pres_approval_avg_x_incpres,ics_x_incpres,no_dem_cand_flag,no_rep_cand_flag
0,2018,Alabama,AL,False,AL-01,89226.0,153228.0,242617,Robert Kennedy Jr,Bradley Byrne,...,-16.192783,-14.281877,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,False,False
1,2018,Alabama,AL,False,AL-02,86931.0,138879.0,226230,Tabitha Isner,Martha Roby,...,-17.411483,-15.288962,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,False,False
2,2018,Alabama,AL,False,AL-03,83996.0,147770.0,231915,Mallory Hagan,Mike Rogers,...,-18.001999,-14.821901,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,False,False
3,2018,Alabama,AL,False,AL-04,46492.0,184255.0,230969,Lee Auman,Robert Aderholt,...,-33.277483,-27.698357,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,False,False
4,2018,Alabama,AL,False,AL-05,101388.0,159063.0,260673,Peter Joffrion,Mo Brooks,...,-18.480031,-16.661366,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,False,False


In [223]:
for col in ['dem_inc', 'rep_inc']:
    mdata[col] = mdata[col].map(lambda x: ast.literal_eval(x))
mdata['dem_inc_any'] = mdata['dem_inc'].map(lambda x: x if not isinstance(x, list) else (True if True in x else False))
mdata['rep_inc_any'] = mdata['rep_inc'].map(lambda x: x if not isinstance(x, list) else (True if True in x else False))

In [224]:
mdata['dem_inc_dummy'] = mdata['dem_inc_any'].map(lambda x: 1 if x == True else 0)
mdata['rep_inc_dummy'] = mdata['rep_inc_any'].map(lambda x: 1 if x == True else 0)

In [225]:
mdata = mdata[mdata['state_po'] != 'DC']

In [228]:
mdata['pvi'] = mdata['prev_lean'] * 0.75 + mdata['prev2_lean'] * 0.25
mdata.head()

,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,...,ics,pres_approval_avg_x_incpres,ics_x_incpres,no_dem_cand_flag,no_rep_cand_flag,dem_inc_any,rep_inc_any,dem_inc_dummy,rep_inc_dummy,pvi
0,2018,Alabama,AL,False,AL-01,89226.0,153228.0,242617,Robert Kennedy Jr,Bradley Byrne,...,97.5,10.073704,-97.5,False,False,False,True,0,1,-15.715056
1,2018,Alabama,AL,False,AL-02,86931.0,138879.0,226230,Tabitha Isner,Martha Roby,...,97.5,10.073704,-97.5,False,False,False,True,0,1,-16.880853
2,2018,Alabama,AL,False,AL-03,83996.0,147770.0,231915,Mallory Hagan,Mike Rogers,...,97.5,10.073704,-97.5,False,False,False,True,0,1,-17.206974
3,2018,Alabama,AL,False,AL-04,46492.0,184255.0,230969,Lee Auman,Robert Aderholt,...,97.5,10.073704,-97.5,False,False,False,True,0,1,-31.882701
4,2018,Alabama,AL,False,AL-05,101388.0,159063.0,260673,Peter Joffrion,Mo Brooks,...,97.5,10.073704,-97.5,False,False,False,True,0,1,-18.025364


In [229]:
mdata.shape

(1521, 37)

In [230]:
mdata.to_csv('transformed/all_2p_house_races_trainset.csv')